# Lumina-Edu - QLoRA fine-tune of NCAIR1/N-ATLaS (Powered by Awarri)

**NAIC 2026 | Problem Statement 3: Sectoral Fine-Tuning (Education & Learning)**

Base model: [`NCAIR1/N-ATLaS`](https://huggingface.co/NCAIR1/N-ATLaS)

This notebook runs the full training pipeline in four stages:

1. **Environment + authentication**
2. **Language capability smoke test** - verifies the model can actually produce
   Yoruba/Hausa/Igbo before we spend GPU hours on it
3. **Self-distillation** - generates genuine translation targets with the base
   model itself
4. **QLoRA fine-tune**

### Before running
- Enable **GPU** (T4 x2 recommended) in notebook settings
- Enable **Internet** so gated N-ATLaS weights can download
- Add `HF_TOKEN` under **Add-ons > Secrets** (needs access to NCAIR1/N-ATLaS)
- Attach the project as a Kaggle dataset, e.g. `/kaggle/input/lumina-edu-data`


## Stage 0 - Environment and authentication

In [ ]:
import os, sys, json, subprocess, importlib.util

HF_TOKEN = os.environ.get("HF_TOKEN") or os.environ.get("KAGGLE_HF_TOKEN")
if not HF_TOKEN:
    try:
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception as e:
        print("No HF_TOKEN in secrets:", e)

assert HF_TOKEN, "Add HF_TOKEN via Add-ons > Secrets"

from huggingface_hub import login
login(token=HF_TOKEN, add_to_git_credential=False)
print("Logged in to Hugging Face")

In [ ]:
!pip install -q unsloth
!pip install -q unsloth_zoo
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q rouge-score scikit-learn lxml pandas

import torch
print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())
print("gpus:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"  gpu{i}: {p.name} {p.total_memory/1e9:.1f}GB")
assert torch.cuda.is_available(), "Enable GPU in notebook settings"

In [ ]:
DATA_ROOT = "/kaggle/input/lumina-edu-data"
if not os.path.exists(DATA_ROOT):
    DATA_ROOT = "/kaggle/working"
    print("WARNING: project dataset not attached; falling back to /kaggle/working")
print("DATA_ROOT =", DATA_ROOT)
assert os.path.exists(os.path.join(DATA_ROOT, "scripts")), "scripts/ not found"
print(sorted(os.listdir(os.path.join(DATA_ROOT, "data/processed"))))

## Stage 1 - Language capability smoke test

The base model's own card rates Yoruba at **2.69/5**. If it cannot produce
target-language text at all, self-distillation is not viable and we must fall
back to English-only training. This check costs ~2 minutes and prevents
committing hours of GPU time to a doomed approach.

In [ ]:
src = open(os.path.join(DATA_ROOT, "scripts/smoke_test_language.py")).read()
exec(compile(src, "smoke_test_language.py", "exec"))

## Stage 2 - Self-distillation

Generates real Yoruba/Hausa/Igbo targets using N-ATLaS itself as teacher.
Output is filtered: samples where the model replied in English or collapsed
into repetition are discarded rather than trained on.

`--limit 3000` costs roughly 4-6 GPU hours. Reduce it if the deadline is tight.

In [ ]:
DISTILL_LIMIT = 3000  # reduce if short on time

rc = subprocess.run([
    sys.executable,
    os.path.join(DATA_ROOT, "scripts/self_distill_translations.py"),
    "--limit", str(DISTILL_LIMIT),
    "--input", os.path.join(DATA_ROOT, "data/processed/train.jsonl"),
    "--out", os.path.join(DATA_ROOT, "data/distilled"),
]).returncode
print("distill exit code:", rc)

In [ ]:
rep_path = os.path.join(DATA_ROOT, "data/distilled/distillation_report.json")
if os.path.exists(rep_path):
    print(json.dumps(json.load(open(rep_path)), indent=2, ensure_ascii=False))
    print("\n>>> REVIEW data/distilled/audit_sample.json BEFORE training.")
    print(">>> These are teacher outputs; a fluent speaker must check quality.")
else:
    print("No distillation report - falling back to English-only training.")

## Stage 3 - Load trainer module and distilled data

In [ ]:
spec = importlib.util.spec_from_file_location(
    "trainer", os.path.join(DATA_ROOT, "scripts/train_lumina_edu.py"))
trainer = importlib.util.module_from_spec(spec)
spec.loader.exec_module(trainer)

DISTILLED = os.path.join(DATA_ROOT, "data/distilled/translations.jsonl")
distilled = trainer.load_jsonl(DISTILLED) if os.path.exists(DISTILLED) else []
print(f"distilled translations: {len(distilled)}")
if distilled:
    from collections import Counter
    print(Counter(d["language"] for d in distilled))

## Stage 4 - QLoRA fine-tune

In [ ]:
trainer.main()

## Next steps

Run the benchmark (500+ samples, real generations vs base model):

```bash
python scripts/evaluate_benchmark.py --limit 500 \
    --adapter lumina-edu-natlas-adapter-powered-by-awarri
```

Artefacts: `data/benchmark/naic_benchmark_results.csv` and
`naic_benchmark_summary.json`.
